# Кластеризация изображений транспортных средств


## Постановка задачи


<center> <img src=https://i.ibb.co/t8DvkyB/smart-city-image-1.jpg align="right" width="300"/> </center>
<center> <img src=https://i.ibb.co/qYkWNVh/smart-city-image-3.jpg align="right" width="300"/> </center>


Один из ключевых проектов IntelliVision — Smart City/Transportation, система, обеспечивающая безопасность дорожного движения и более эффективную работу парковок. С помощью Smart City/Transportation можно контролировать сигналы светофоров и соблюдение ограничений скорости, определять виды транспортных средств, распознавать номерные знаки, считать автомобили и людей.

В основе всех перечисленных возможностей проекта лежит CV (Computer Vision, компьютерное зрение). Чтобы их реализовать, компания использует модели, для обучения которых применяются огромные размеченные датасеты с изображениями транспортных средств. Однако система работает в режиме реального времени и с каждым днём данных становится всё больше. Алгоритм нуждается в постоянной модернизации и должен учитывать множество факторов.

Для модификации и повышения эффективности системы Smart City/Transportation команде необходимо автоматизировать определение дополнительных параметров авто на изображении:

* тип автомобиля (кузова),
* ракурс снимка (вид сзади/спереди),
* цвет автомобиля,
* другие характеристики.

Также необходимо автоматизировать поиск выбросов в данных (засветы и блики на изображениях, изображения, на которых отсутствуют автомобили и т. д.).

К сожалению, у компании нет комплексной модели, которая могла бы одновременно находить на изображении автомобиль и определять все нужные параметры. Её нужно построить, однако многокомпонентная разметка новых данных по всем этим параметрам — очень трудозатратное занятие, которое стоит больших денег.

При решении задачи разметки данных у команды возникла гипотеза, которая нуждается в исследовании.


**Гипотеза:** разметку исходных данных можно эффективно провести с помощью методов кластеризации. 


**В чём идея?**

*Давайте будем использовать небольшой набор моделей свёрточных нейронных сетей, обученных на различных датасетах и решающих различные задачи от классификации изображений по цвету до классификации типов транспортных средств, пропустим нашу базу изображений через каждую модель, но возьмём не выходной результат модели, а только промежуточное представление признаков (дескриптор), полученное на свёрточных слоях сети.*

*Выполним такую операцию для всех изображений из набора данных, на основе полученных дескрипторов кластеризуем изображения, проинтерпретируем полученные кластеры и попробуем найти в них необходимую информацию.*

Теперь, когда мы обсудили гипотезу, перейдём к постановке задачи.

<center> <img src=https://i.ibb.co/hLcBpZF/2023-03-27-12-11-17.png align="right" width="500"/> </center>

В кейсе используется набор из 416 314 изображений транспортных средств различных типов, цветов и снятых с разных ракурсов.

Команда IntelliVision уже обработала свой набор данных с помощью нескольких моделей глубокого обучения (свёрточных нейронных сетей) и получила четыре варианта вектора признаков (дескрипторов) для каждого изображения.

**Цель исследования** — используя готовые дескрипторы, разбить изображения на кластеры и проинтерпретировать каждый из них. Для всех вариантов дескрипторов нужно применить несколько алгоритмов кластеризации и сравнить полученные результаты. Сравнивать можно на основе метрик, визуализаций плотностей кластеров и по тому, насколько хорошо интерпретируются кластеры.

Дополнительная подзадача — найти выбросы среди изображений. Это могут быть изображения плохого качества, изображения с бликами или изображения, на которых нет транспортных средств и т. д.

Бизнес-задача: исследовать возможность применения алгоритмов кластеризации для разметки новых данных и поиска выбросов.

Техническая задача для вас как для специалиста в Data Science: построить модель кластеризации изображений на основе дескрипторов, выделяемых с помощью различных архитектур нейронных сетей, проинтерпретировать полученные результаты и выбрать модель или комбинацию моделей, которая выделяет наиболее пригодные для интерпретации признаки.

**Ваши основные цели:**
1. Для каждого типа дескрипторов необходимо:
    * выполнить предобработку дескрипторов;
    * произвести кластеризацию изображений на основе их дескрипторов, подобрав алгоритм и параметры кластеризации;
    * сделать визуализацию полученных кластеров в 2D- или 3D-пространстве;
    * проинтерпретировать полученные кластеры — в паре предложений сформулировать, какие изображения попали в каждый из кластеров.
2. Сравнить между собой полученные кластеризации для каждого типа дескрипторов (по метрикам, визуализации и результатам интерпретации).
3. Выполнить автоматизированный поиск выбросов среди изображений на основе дескрипторов.
4. Дополнительно (по желанию): попробовать воспользоваться смесью дескрипторов, полученных различными моделями, и проинтерпретировать полученные результаты.

**Примечание.** При выборе алгоритма кластеризации следует ориентироваться на внутренние метрики, а именно на индекс Калински — Харабаса (`calinski_harabasz_score`) и индекс Дэвиса — Болдина (`davies_bouldin_score`), а также на интерпретируемость кластеров и визуализацию.


## Данные и их описание


Структура папки `data/` в корне репозитория:

```
data/
├── descriptors/
│   ├── efficientnet-b7.pickle
│   ├── osnet.pickle
│   ├── vdc_color.pickle
│   └── vdc_type.pickle
├── raw_data/
│   └── veriwild/
│       └── ...
├── images_paths.csv
└── clustering_results_best.csv
```

Давайте разберёмся в ней:

* В папке `descriptors` содержатся дескрипторы, полученные для каждого из изображений с помощью соответствующих нейронных сетей, в формате numpy-массивов, сохранённых в файлах pickle:
    * `efficientnet-b7.pickle` — дескрипторы, выделенные моделью классификации с архитектурой EfficientNet версии 7. Эта модель является свёрточной нейронной сетью, предобученной на датасете ImageNet, в котором содержатся изображения более 1000 различных классов. Эта модель при обучении не видела датасета veriwild. 

    * `osnet.pickle` — дескрипторы, выделенные моделью OSNet, обученной для детектирования людей, животных и машин. Модель не обучалась на исходном датасете veriwild.

    * `vdc_color.pickle` — дескрипторы, выделенные моделью регрессии для определения цвета транспортных средств в формате RGB. Частично обучена на исходном датасете veriwild.
    
    * `vdc_type.pickle` — дескрипторы, выделенные моделью классификации транспортных средств по типу на десяти классах. Частично обучена на исходном датасете veriwild.

* В папке `raw_data` содержится zip-архив с исходными изображениями автомобилей. Распакуйте его содержимое в папку raw_data. Архив содержит десять папок с изображениями, пронумерованных от 1 до 10. Каждая папка содержит подпапки, обозначенные пятизначными цифрами, например 36191. 

В каждой из таких подпапок содержатся фотографии одного конкретного автомобиля с разных ракурсов, снятые с помощью дорожных видеокамер.

* В файле `images_paths.csv` представлен список из полных путей до изображений. Он пригодится вам при анализе изображений, попавших в определённый кластер.


Импорт базовых библиотек:


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings 

from IPython.display import display, HTML

warnings.filterwarnings("ignore")

plt.rcParams["patch.force_edgecolor"] = True 


## 1. Знакомство со структурой данных


Загрузите numpy-массивы из pickle-файлов в `data/descriptors/`.

**Примечание** Для удобства дальнейшей работы вы можете составить четыре DataFrame с путями до изображений и соответствующими им дескрипторами.

Посмотрите на размерности каждой из четырёх заданных матриц и сравните использованные модели глубокого обучения по размерностям выходных дескрипторов изображений. 


In [ ]:
import gc
import pickle
from pathlib import Path

from tqdm.auto import tqdm

# Настройка отображения
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

BASE_DIR = Path('..') / 'data'

# --- Подвыборка: задаётся до загрузки дескрипторов (экономия RAM) ---
# None = все объекты (нужно много памяти). При нехватке RAM уменьшите, например, до 30_000.
SUBSET_SIZE = 80_000
SUBSET_SEED = 42

paths_file = BASE_DIR / 'images_paths.csv'
path_col = pd.read_csv(paths_file, nrows=0).columns[0]
paths_series = pd.read_csv(paths_file, usecols=[path_col])[path_col].astype(str)
n_total = len(paths_series)
print(f"Всего путей в датасете: {n_total:,}")

if SUBSET_SIZE is not None and n_total > SUBSET_SIZE:
    idx = np.sort(np.random.RandomState(SUBSET_SEED).choice(n_total, SUBSET_SIZE, replace=False))
    paths = paths_series.iloc[idx].tolist()
    print(f"Рабочая подвыборка: {len(paths):,} объектов (seed={SUBSET_SEED})")
else:
    idx = None
    paths = paths_series.tolist()
    print(f"Используются все объекты: {len(paths):,}")

del paths_series
display(pd.DataFrame({'paths': paths[:3]}))

descriptors_dir = BASE_DIR / 'descriptors'
pickle_files = [
    'efficientnet-b7.pickle',
    'osnet.pickle',
    'vdc_color.pickle',
    'vdc_type.pickle',
]

dfs = {}
for pkl_name in tqdm(pickle_files, desc="Загрузка дескрипторов"):
    name = pkl_name.replace('.pickle', '')
    with open(descriptors_dir / pkl_name, 'rb') as f:
        arr = np.asarray(pickle.load(f), dtype=np.float32)

    if arr.shape[0] != n_total:
        raise ValueError(
            f"{name}: строк в pickle ({arr.shape[0]:,}) != путей в CSV ({n_total:,})"
        )

    if idx is not None:
        arr = arr[idx]
    # DataFrame только по подвыборке; полный массив освобождаем сразу
    df = pd.DataFrame(arr, copy=False)
    df.insert(0, 'paths', paths)
    dfs[name] = df
    del arr
    gc.collect()
    print(f"{name}: в работе {df.shape[0]:,} × {df.shape[1] - 1} признаков (float32)")

dims = [(name, dfs[name].shape[1] - 1, dfs[name].shape[0], n_total) for name in dfs]
summary = pd.DataFrame(
    dims,
    columns=['Модель (дескриптор)', 'Размерность', 'Объектов в работе', 'Всего в датасете'],
)
display(summary)


**Первичные выводы о структуре данных**

- Во всех четырёх pickle число строк совпадает с `images_paths.csv` (полный датасет — 416 314 объектов); в память загружается только подвыборка `SUBSET_SIZE` (см. ячейку загрузки).
- Размерности векторов дескрипторов различаются в зависимости от модели: у **EfficientNet-B7** — наибольшая (2560 признаков), у **OSNet** и **vdc_type** — по 512 признаков, у **vdc_color** — 128 признаков. Это связано с разной архитектурой сетей и размером латентного представления.
- Для дальнейшей работы используются четыре DataFrame в словаре `dfs` (ключи: `efficientnet-b7`, `osnet`, `vdc_color`, `vdc_type`); в каждом первый столбец — `paths`, остальные — признаки дескриптора.


## 2. Преобразование, очистка и анализ данных


Признаки, найденные с помощью некоторых моделей, исчисляются тысячами, что довольно много, учитывая общее количество наблюдений.

Как вы понимаете, производить кластеризацию на таком большом количестве признаков, которые были сформированы исходными моделями глубокого обучения, довольно сложно и затратно по времени. К тому же, многие признаки, найденные моделями на изображениях, могут быть сильно скоррелированы между собой.

Понизьте размерность исходных дескрипторов с помощью соответствующих методов. Можно уменьшить размерность входных данных до 100 или 200 признаков — этого будет достаточно, чтобы произвести кластеризацию, однако рекомендуем вам самостоятельно подобрать необходимое количество компонент в новом пространстве признаков.

Также позаботьтесь о масштабе признаков, воспользовавшись стандартизацией и нормализацией. После кластеризации определите, какой вариант масштабирования более успешен для каждого варианта дескрипторов.


In [ ]:
# =============================================================================
# Предобработка: масштабирование (StandardScaler, MinMaxScaler) и PCA по батчам.
# Число компонент — по порогу объяснённой дисперсии (90%), не более 200.
# =============================================================================

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA, IncrementalPCA

VARIANCE_THRESHOLD = 0.90
MAX_COMPONENTS = 200
SAMPLE_SIZE = 50_000
BATCH_SIZE = 20_000


def _get_n_components(X_sample_scaled, variance_threshold, max_components):
    """Число компонент PCA: столько, чтобы кумулятивная дисперсия >= variance_threshold, но не больше max_components."""
    n_features = X_sample_scaled.shape[1]
    n_comp = min(max_components, n_features - 1)
    if n_comp <= 0:
        return 1
    pca = PCA(n_components=n_comp, random_state=42)
    pca.fit(X_sample_scaled)
    cumvar = np.cumsum(pca.explained_variance_ratio_)
    n = np.searchsorted(cumvar, variance_threshold) + 1
    return min(n, max_components, n_features)


def reduce_and_scale(dfs_dict, variance_threshold=0.90, max_components=200):
    """
    Для каждого дескриптора: два масштабирования (std, minmax), затем PCA по батчам (IncrementalPCA).
    Scaler и n_components подбираются по выборке; полные данные обрабатываются батчами с прогрессом.
    """
    results = {}
    names = list(dfs_dict.keys())

    for name_idx, name in enumerate(names, start=1):
        print(f"\n[{name_idx}/{len(names)}] Дескриптор: {name}")
        df = dfs_dict[name]
        paths = df['paths'].copy()
        n_rows = len(df)
        n_features = df.drop(columns=['paths']).shape[1]

        n_sample = min(SAMPLE_SIZE, n_rows)
        idx = np.random.RandomState(42).choice(n_rows, n_sample, replace=False)
        X_sample = df.drop(columns=['paths']).iloc[idx].values.astype(np.float32)

        results[name] = {}
        for scale_idx, (scale_name, ScalerClass) in enumerate([('std', StandardScaler), ('minmax', MinMaxScaler)], start=1):
            print(f"  Масштабирование: {scale_name} ({scale_idx}/2)")
            scaler = ScalerClass()
            X_sample_scaled = scaler.fit_transform(X_sample)
            n_components = _get_n_components(X_sample_scaled, variance_threshold, max_components)
            ipca = IncrementalPCA(n_components=n_components, batch_size=BATCH_SIZE)

            batch_starts = list(range(0, n_rows, BATCH_SIZE))
            for start in tqdm(batch_starts, desc="  partial_fit", leave=False):
                end = min(start + BATCH_SIZE, n_rows)
                batch = df.drop(columns=['paths']).iloc[start:end].values.astype(np.float32)
                ipca.partial_fit(scaler.transform(batch))

            parts = []
            for start in tqdm(batch_starts, desc="  transform", leave=False):
                end = min(start + BATCH_SIZE, n_rows)
                batch = df.drop(columns=['paths']).iloc[start:end].values.astype(np.float32)
                parts.append(ipca.transform(scaler.transform(batch)))
            X_reduced = np.vstack(parts).astype(np.float32)

            cumvar = np.cumsum(ipca.explained_variance_ratio_)[-1]
            results[name][scale_name] = {
                'X': X_reduced,
                'paths': paths,
                'scaler': scaler,
                'pca': ipca,
                'n_components': n_components,
                'explained_variance_ratio': ipca.explained_variance_ratio_,
                'cumulative_variance': float(cumvar),
            }

        n_std = results[name]['std']['n_components']
        cumvar_std = results[name]['std']['cumulative_variance']
        print(f"  Итого: {n_std} компонент, объяснённая дисперсия {cumvar_std:.3f}")

    return results


print("Старт предобработки...")
preprocessed = reduce_and_scale(dfs, variance_threshold=VARIANCE_THRESHOLD, max_components=MAX_COMPONENTS)
print("\nПредобработка завершена.") 


In [ ]:
# Графики кумулятивной объяснённой дисперсии для обоснования выбора числа компонент.
# Красная линия — порог 90%.

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
for idx, name in enumerate(preprocessed):
    pca = preprocessed[name]['std']['pca']
    n_comp = pca.n_components_
    cumvar = np.cumsum(pca.explained_variance_ratio_)
    axes[idx].plot(np.arange(1, n_comp + 1), cumvar, 'b-')
    axes[idx].axhline(y=VARIANCE_THRESHOLD, color='red', linestyle='--', label=f'{VARIANCE_THRESHOLD*100:.0f}%')
    axes[idx].set_xlabel('Число компонент')
    axes[idx].set_ylabel('Кумулятивная объяснённая дисперсия')
    axes[idx].set_title(f'{name} (выбрано {n_comp} компонент)')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)
plt.tight_layout()
plt.show() 


**Обоснование выбора числа компонент и масштабирования**

- Для понижения размерности использован **PCA**. Число компонент выбрано так, чтобы кумулятивная объяснённая дисперсия была не ниже **90%** (параметр `VARIANCE_THRESHOLD = 0.90`), при этом не более **200** компонент (ограничение по заданию и вычислительным затратам).
- На графиках выше по горизонтали — число компонент, по вертикали — доля объяснённой дисперсии. Красная линия — порог 90%. Выбранное число компонент обеспечивает сохранение достаточной информации для кластеризации при сокращении размерности.
- Для каждого набора дескрипторов получены два варианта предобработки: **стандартизация (StandardScaler)** и **нормализация (MinMaxScaler)**. Итоговое качество кластеризации для этих вариантов будет сравнено в разделе 3, чтобы определить, какой вариант масштабирования более успешен для каждого типа дескрипторов.


**Вывод**

- Для ускорения экспериментов использована подвыборка из 80 000 объектов (фиксированный seed 42). Для каждого из четырёх типов дескрипторов выполнено масштабирование (StandardScaler и MinMaxScaler) и понижение размерности методом PCA. Число компонент подбиралось по порогу 90% кумулятивной объяснённой дисперсии с верхним ограничением 200 компонент.

- По результатам предобработки: для **osnet** выбрано 153 компоненты (объяснённая дисперсия 90,0%), для **vdc_color** — 69 компонент (90,2%), для **vdc_type** — 20 компонент (90,1%). Для **efficientnet-b7** достигнуто ограничение в 200 компонент при объяснённой дисперсии 62,7%: исходная размерность (2560) велика, и для выхода на 90% потребовалось бы больше компонент; верхняя граница 200 принята по заданию и вычислительным затратам, оставшейся доли дисперсии достаточно для последующей кластеризации.

- Результаты сохранены в словаре `preprocessed`: для каждого дескриптора доступны варианты со стандартизацией (`std`) и нормализацией (`minmax`) с матрицей признаков `X` и путями `paths`. В разделе 3 на этих данных будет выполнена кластеризация и сравнение вариантов масштабирования по метрикам Калински — Харабаса и Дэвиса — Болдина.


## 3. Моделирование и оценка качества модели


### 3.1. Кластеризация изображений


После предобработки исходных данных произведите кластеризацию для каждого набора дескрипторов.

Для решения задачи используйте несколько различных методов, подобрав оптимальное количество кластеров для каждого метода и варианта дескрипторов.

В качестве метрики для подбора оптимального количества кластеров используйте внутренние меры индекс Калински — Харабаса (`calinski_harabasz_score`) и индекс Дэвиса — Болдина (`davies_bouldin_score`).

Рекомендуем вынести код для построения моделей кластеризации и подбора их параметров в отдельную функцию, чтобы не множить одинаковый код для четырёх случаев дескрипторов.

**Примечание.** Поскольку исходных данных много, могут возникнуть проблемы с оперативной памятью и скоростью работы таких алгоритмов, как K-Means. Вместо стандартного алгоритма K-Means можно воспользоваться реализацией MiniBatchKMeans. 

**Примечание.** Постарайтесь написать чистый код, максимально уменьшая количество дублирующихся участков.


In [ ]:
# =============================================================================
# 3.1. Кластеризация изображений
# Для каждого дескриптора: несколько алгоритмов, подбор оптимального k по CH и DB.
# Алгоритмы: MiniBatchKMeans, GaussianMixture, AgglomerativeClustering (сетка по выборке).
# =============================================================================

from sklearn.cluster import MiniBatchKMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score

# Диапазон числа кластеров для подбора
K_RANGE = [5, 10, 15, 20, 25, 30]
# Размер батча для MiniBatchKMeans
BATCH_SIZE_KMEANS = 5000
# Размер выборки для сетки по k в AgglomerativeClustering (экономия памяти и времени)
AGGLOMERATIVE_SAMPLE_SIZE = 15_000
# Используемый вариант масштабирования для кластеризации 
SCALING_USED = 'std'


def run_clustering_for_descriptor(X, descriptor_name):
    """
    Запускает три алгоритма кластеризации, подбирает k по CH (макс.) и DB (мин.).
    Для AgglomerativeClustering сетка по k выполняется на выборке, затем
    один раз подгонка на полных данных с лучшим k (или назначение по центроидам выборки).
    Возвращает словарь: имя_алгоритма -> {labels, best_k, ch_score, db_score}.
    """
    n_samples = X.shape[0]
    results = {}

    # --- MiniBatchKMeans ---
    best_ch, best_db, best_k = -np.inf, np.inf, K_RANGE[0]
    for k in tqdm(K_RANGE, desc="  MiniBatchKMeans", leave=False):
        model = MiniBatchKMeans(n_clusters=k, random_state=42, batch_size=BATCH_SIZE_KMEANS)
        labels = model.fit_predict(X)
        ch = calinski_harabasz_score(X, labels)
        db = davies_bouldin_score(X, labels)
        if ch > best_ch:
            best_ch, best_db, best_k = ch, db, k
    model_final = MiniBatchKMeans(n_clusters=best_k, random_state=42, batch_size=BATCH_SIZE_KMEANS)
    labels_kmeans = model_final.fit_predict(X)
    results['MiniBatchKMeans'] = {
        'labels': labels_kmeans,
        'best_k': best_k,
        'ch_score': calinski_harabasz_score(X, labels_kmeans),
        'db_score': davies_bouldin_score(X, labels_kmeans),
    }

    # --- GaussianMixture ---
    best_ch, best_db, best_k = -np.inf, np.inf, K_RANGE[0]
    for k in tqdm(K_RANGE, desc="  GaussianMixture", leave=False):
        model = GaussianMixture(n_components=k, covariance_type='diag', random_state=42)
        labels = model.fit_predict(X)
        ch = calinski_harabasz_score(X, labels)
        db = davies_bouldin_score(X, labels)
        if ch > best_ch:
            best_ch, best_db, best_k = ch, db, k
    model_final = GaussianMixture(n_components=best_k, covariance_type='diag', random_state=42)
    labels_gmm = model_final.fit_predict(X)
    results['GaussianMixture'] = {
        'labels': labels_gmm,
        'best_k': best_k,
        'ch_score': calinski_harabasz_score(X, labels_gmm),
        'db_score': davies_bouldin_score(X, labels_gmm),
    }

    # --- AgglomerativeClustering: сетка по k на выборке, затем разметка всех по центроидам выборки ---
    n_agg_sample = min(AGGLOMERATIVE_SAMPLE_SIZE, n_samples)
    rng = np.random.RandomState(42)
    idx_agg = rng.choice(n_samples, n_agg_sample, replace=False)
    X_agg_sample = X[idx_agg]
    best_ch, best_k = -np.inf, K_RANGE[0]
    for k in tqdm(K_RANGE, desc="  Agglomerative (сетка по выборке)", leave=False):
        model = AgglomerativeClustering(n_clusters=k)
        labels_s = model.fit_predict(X_agg_sample)
        ch = calinski_harabasz_score(X_agg_sample, labels_s)
        if ch > best_ch:
            best_ch, best_k = ch, k
    model_agg = AgglomerativeClustering(n_clusters=best_k)
    labels_agg_sample = model_agg.fit_predict(X_agg_sample)
    # Центроиды кластеров по выборке
    centers = np.array([X_agg_sample[labels_agg_sample == i].mean(axis=0) for i in range(best_k)])
    # Назначение всех объектов по ближайшему центроиду
    from scipy.spatial.distance import cdist
    dists = cdist(X, centers, metric='euclidean')
    labels_agg = np.argmin(dists, axis=1)
    results['AgglomerativeClustering'] = {
        'labels': labels_agg,
        'best_k': best_k,
        'ch_score': calinski_harabasz_score(X, labels_agg),
        'db_score': davies_bouldin_score(X, labels_agg),
    }

    return results


# Запуск по всем дескрипторам (используем вариант масштабирования SCALING_USED)
clustering_results = {}
names = list(preprocessed.keys())

print("Старт кластеризации (для каждого дескриптора: 3 алгоритма, подбор k)...")
for idx, name in enumerate(names, start=1):
    print(f"\n[{idx}/{len(names)}] Дескриптор: {name}")
    X = preprocessed[name][SCALING_USED]['X']
    clustering_results[name] = run_clustering_for_descriptor(X, name)
print("\nКластеризация завершена.") 


In [ ]:
# =============================================================================
# Дополнительная кластеризация для варианта предобработки MinMaxScaler (minmax)
# Используем ту же функцию run_clustering_for_descriptor и те же алгоритмы.
# =============================================================================

clustering_results_minmax = {}

print("Старт кластеризации (minmax) для сравнения предобработки...")
names = list(preprocessed.keys())
for idx, name in enumerate(names, start=1):
    print(f"\n[{idx}/{len(names)}] Дескриптор: {name} (minmax)")
    X_minmax = preprocessed[name]['minmax']['X']
    clustering_results_minmax[name] = run_clustering_for_descriptor(X_minmax, name)
print("\nКластеризация (minmax) завершена.")


In [ ]:
# =============================================================================
# Сводная таблица: дескриптор, алгоритм, лучшее k, Калински — Харабаса, Дэвиса — Болдина
# Сравнение std vs minmax по метрикам кластеризации
# =============================================================================

rows = []

# std
for desc_name, algo_dict in clustering_results.items():
    for algo_name, res in algo_dict.items():
        rows.append({
            'Дескриптор': desc_name,
            'Алгоритм': algo_name,
            'Масштабирование': 'std',
            'k': res['best_k'],
            'Калински — Харабаса': round(res['ch_score'], 2),
            'Дэвиса — Болдина': round(res['db_score'], 4),
        })

# minmax
for desc_name, algo_dict in clustering_results_minmax.items():
    for algo_name, res in algo_dict.items():
        rows.append({
            'Дескриптор': desc_name,
            'Алгоритм': algo_name,
            'Масштабирование': 'minmax',
            'k': res['best_k'],
            'Калински — Харабаса': round(res['ch_score'], 2),
            'Дэвиса — Болдина': round(res['db_score'], 4),
        })

table_clustering = pd.DataFrame(rows)
display(table_clustering) 


**Метрики и интерпретация**

- **Калински — Харабаса:** выше — лучше (больше разделённость кластеров относительно разброса внутри).
- **Дэвиса — Болдина:** ниже — лучше (меньше отношение внутрикластерного разброса к межкластерному).

**Алгоритмы**

- **MiniBatchKMeans:** быстрый, масштабируется на большие выборки; требует задавать k; чувствителен к форме кластеров (сферические).
- **GaussianMixture:** мягкие метки, устойчив к выбросам при диагональной ковариации; требует задавать k; дольше K-Means.
- **AgglomerativeClustering:** иерархия кластеров; для экономии времени сетка по k выполнена на выборке, разметка полной выборки — по ближайшему центроиду, полученному по этой выборке. 


**Сравнение вариантов предобработки (StandardScaler vs MinMaxScaler)**

- Для каждого дескриптора и алгоритма была выполнена кластеризация как на стандартизированных признаках (`std`), так и на признаках после нормализации (`minmax`) при одинаковом числе кластеров (k=5). Сводная таблица `table_clustering` показывает, что во всех случаях при переходе от `std` к `minmax` индекс Калински — Харабаса **увеличивается**, то есть разделённость кластеров относительно разброса внутри них становится выше.

- Для дескрипторов **efficientnet-b7** и **osnet** переход к `minmax` улучшает обе метрики: и Калински — Харабаса (выше), и Дэвиса — Болдина (ниже). Для **vdc_color** и **vdc_type** нормализация `minmax` даёт более высокие значения Калински — Харабаса, но индекс Дэвиса — Болдина немного увеличивается (кластеры становятся более разделёнными, но слегка возрастает внутрикластерный разброс в пересчёте на расстояния между кластерами).

- Визуализация и интерпретация кластеров в дальнейшем выполнялись на варианте **StandardScaler + PCA**, так как уже при этом варианте качество кластеризации оказалось достаточным, а результаты `minmax` по характеру кластеров и т-SNE дают схожую картину. При необходимости более агрессивной оптимизации внутренних метрик можно использовать вариант `minmax` (особенно для efficientnet-b7 и osnet), сохраняя ту же схему подбора параметров.


**Вывод**

- Для всех четырёх дескрипторов и трёх алгоритмов (MiniBatchKMeans, GaussianMixture, AgglomerativeClustering) оптимальное число кластеров по метрикам Калински — Харабаса и Дэвиса — Болдина оказалось равным 5 (нижняя граница заданного диапазона). Это говорит о том, что в пространстве признаков формируется несколько крупных и устойчивых групп.

- По дескрипторам наилучшие значения метрик даёт пара **vdc_type** и **vdc_color**: для них значения Калински — Харабаса максимальны, а Дэвиса — Болдина — минимальны среди всех дескрипторов. **osnet** показывает среднее качество, **efficientnet-b7** — наихудшее. Специализированные дескрипторы (тип и цвет ТС) формируют более разделимые кластеры, чем универсальные признаки общих сетей.

- По алгоритмам в большинстве случаев лучшим оказывается **AgglomerativeClustering**: он даёт наибольшие значения Калински — Харабаса и наименьшие Дэвиса — Болдина для efficientnet-b7, osnet и vdc_type; для vdc_color он также имеет минимальный индекс Дэвиса — Болдина при CH, сопоставимом с MiniBatchKMeans. MiniBatchKMeans даёт результаты чуть хуже, GaussianMixture — ещё ниже по CH и выше по DB. Далее в работе для визуализации и интерпретации кластеров использовались результаты AgglomerativeClustering.


### 3.2. Интерпретация кластеров


#### 3.2.1 Визуализация кластеров


Визуализируйте результаты кластеризации в двух- или трёхмерном пространстве, предварительно понизив размерность дескрипторов изображений до соответствующих размерностей с помощью метода t-SNE. 

По результатам визуализации кластеров сделайте предположение о качестве полученной кластеризации.


In [ ]:
# =============================================================================
# 3.2.1. Визуализация кластеров в 2D после понижения размерности t-SNE
# Для каждого дескриптора: выборка -> t-SNE(2) -> scatter по меткам лучшего алгоритма.
# =============================================================================

from sklearn.manifold import TSNE

TSNE_SAMPLE_SIZE = 5_000
ALGORITHM_FOR_PLOT = 'AgglomerativeClustering'
SCALING_USED = 'std'

tsne_embeddings = {}

print("Построение 2D t-SNE и scatter по кластерам для каждого дескриптора...")
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, name in enumerate(tqdm(preprocessed.keys(), desc="t-SNE + графики")):
    X = preprocessed[name][SCALING_USED]['X']
    labels = clustering_results[name][ALGORITHM_FOR_PLOT]['labels']
    n = X.shape[0]
    n_sample = min(TSNE_SAMPLE_SIZE, n)
    rng = np.random.RandomState(42)
    sample_idx = rng.choice(n, n_sample, replace=False)
    X_sample = X[sample_idx]
    labels_sample = labels[sample_idx]

    # В актуальном sklearn параметр называется max_iter, не n_iter
    tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
    X_2d = tsne.fit_transform(X_sample)
    tsne_embeddings[name] = (X_2d, labels_sample)

    scatter = axes[idx].scatter(
        X_2d[:, 0], X_2d[:, 1],
        c=labels_sample, cmap='tab10', alpha=0.6, s=8
    )
    axes[idx].set_title(f"{name}\n({ALGORITHM_FOR_PLOT}, выборка {n_sample})")
    axes[idx].set_xlabel("t-SNE 1")
    axes[idx].set_ylabel("t-SNE 2")
    plt.colorbar(scatter, ax=axes[idx], label="Кластер")

plt.tight_layout()
plt.show()
print("Готово.") 


**Вывод**

- По графикам t-SNE видно, что качество кластеризации сильно зависит от дескриптора. У **efficientnet-b7** кластеры почти не разделены: точки разных цветов сильно перемешаны в одной плотной области, границ между группами почти нет. У **vdc_color** картина смешанная: один–два кластера (например, тёмно-красный внизу и синий слева) хорошо обособлены, а в центральной массе группы снова сильно перекрываются.

- У **osnet** разделение лучше: несколько компактных областей (синий слева, красно-коричневый справа сверху, зелёный по центру снизу), но в центре остаётся заметное смешение. У **vdc_type** кластеры выглядят наиболее чётко: зелёный сверху по центру, синий внизу справа, серый слева и другие группы образуют отдельные сгустки с видимыми «просветами» между ними.

- Итог: **vdc_type** даёт наиболее разделимые и интерпретируемые кластеры, **osnet** — среднее качество, **efficientnet-b7** и в большой части **vdc_color** — слабое разделение. Это согласуется с метриками из п. 3.1 (наилучшие CH и DB у vdc_type и vdc_color по таблице, а по визуализации vdc_type лучше vdc_color по степени разделённости).


#### 3.2.2. Визуализация изображений в кластере


Визуализируйте несколько изображений из каждого кластера, чтобы проинтерпретировать результаты.

**Как визуализировать изображения, соответствующие определённому кластеру?**

Мы не рассматривали работу с изображениями как отдельную тему, однако не волнуйтесь — в этом нет ничего страшного.

В стандартных библиотеках для визуализации, которые мы изучали ранее, есть встроенный функционал для чтения и визуализации изображений. Например, в библиотеке matplotlib есть функция `plt.imread()`, которая позволяет читать изображение по переданному пути. Она возвращает numpy-массив размерности (h, w, c), где:

* h — высота изображения, 
* w — его ширина,
* c — количество каналов.

Так как все изображения в нашем датасете цветные, каналов (c) три:

* R — матрица интенсивности пикселей красного цвета,
* G — матрица интенсивности пикселей зелёного цвета,
* B — матрица интенсивности пикселей синего цвета.

Например, вот так можно прочитать изображение 000001.jpg:

```python
img = plt.imread('../data/raw_data/veriwild/1/00001/000001.jpg')
print(img.shape)
## (557, 756, 3)
```

То есть изображение состоит из трёх матриц (R, G и B) с размерностью 557 строк на 756 столбцов. Элементами каждой из матриц являются интенсивности пикселей (от 0 до 255) соответствующего цвета.

Что касается вывода изображений на экран, в библиотеке matplotlib есть встроенная функция `plt.imshow()`, которая позволяет вывести переданное ей в аргументы изображение:

```python
fig = plt.figure(figsize=(5, 5))
plt.imshow(img);
```

Функцию `imshow()` можно вызывать и от имени координатных плоскостей при использовании `subplots` из библиотеки `matplotlib`:

```python
img1 = plt.imread('../data/raw_data/veriwild/1/00001/000001.jpg')
img2 = plt.imread('../data/raw_data/veriwild/1/00001/000002.jpg')
fig, axes = plt.subplots(1, 2, figsize=(5, 5))
axes[0].imshow(img1);
axes[1].imshow(img2);
```

После кластеризации для интерпретации результатов вам понадобится визуализировать несколько изображений из каждого кластера. Для этого мы подготовили функцию `plot_sample_cluster_images()`.


In [ ]:
def plot_samples_images(data, cluster_label, nrows=3, ncols=3, figsize=(12, 5)):
    """Функция для визуализации нескольких случайных изображений из кластера cluster_label.
    Пути до изображений и метки кластеров должны быть представлены в виде DataFrame со столбцами "paths" и "cluster".

    Args:
        data (DataFrame): таблица с разметкой изображений и соответствующих им кластеров.
        cluster_label (int): номер кластера изображений.
        nrows (int, optional): количество изображений по строкам таблицы.
        ncols (int, optional): количество изображений по столбцам.
        figsize (tuple, optional): размер фигуры.
    """
    # Фильтруем данные по номеру кластера
    samples_indexes = np.array(data[data['cluster'] == cluster_label].index)
    # Перемешиваем результаты
    np.random.shuffle(samples_indexes)
    # Составляем пути до изображений
    paths = data.loc[samples_indexes, 'paths']
   
    # Создаём фигуру и набор координатных плоскостей
    fig, axes = plt.subplots(nrows,ncols)
    # Устанавливаем размер фигуры
    fig.set_size_inches(*figsize)
    # Устанавливаем название графика
    fig.suptitle(f"Images from cluster {cluster_label}", fontsize=16)
    # Создаём цикл по строкам в таблице с координатными плоскостями
    for i in range(nrows):
        # Создаём цикл по столбцам в таблице с координатными плоскостями
        for j in range(ncols):
            # Определяем индекс пути до изображения
            path_idx = i * ncols + j
            if path_idx >= len(paths):
                break
            # Извлекаем путь до изображения
            path = paths.iloc[path_idx]
            # Читаем изображение
            img = plt.imread(path)
            # Отображаем его на соответствующей координатной плоскости
            axes[i,j].imshow(img)
            # Убираем пометки координатных осей
            axes[i,j].axis('off')


Например, вы произвели кластеризацию и записали пути до изображений в виде столбца "paths" и метки кластеров в виде столбца "cluster" в некоторый DataFrame с именем data. Тогда, чтобы визуализировать несколько случайных изображений из кластера 0, вам нужно вызвать функцию `plot_sample_cluster_images()` следующим образом:

```python
plot_samples_images(data=data, cluster_label=0)
```


In [ ]:
# =============================================================================
# 3.2.2. Визуализация нескольких изображений из каждого кластера для каждого дескриптора
# В заголовке каждой фигуры: имя дескриптора + номер кластера.
# =============================================================================

BASE_DIR = Path('..') / 'data'
IMAGES_BASE = BASE_DIR / 'raw_data'
ALGORITHM_FOR_PLOT = 'AgglomerativeClustering'
NROWS, NCOLS = 3, 3

np.random.seed(42)

for desc_idx, name in enumerate(tqdm(preprocessed.keys(), desc="Дескрипторы")):
    paths = preprocessed[name]['std']['paths']
    labels = clustering_results[name][ALGORITHM_FOR_PLOT]['labels']
    full_paths = [str(IMAGES_BASE / str(p).replace('\\', '/')) for p in paths]
    data = pd.DataFrame({'paths': full_paths, 'cluster': labels})
    k = int(labels.max()) + 1
    for c in tqdm(range(k), desc=f"  {name}", leave=False):
        plot_samples_images(data, cluster_label=c, nrows=NROWS, ncols=NCOLS, figsize=(12, 5))
        plt.gcf().suptitle(f"{name} — Images from cluster {c}", fontsize=16)
    plt.show()


**Вывод**

**efficientnet-b7:** Кластеры 0, 2 и 4 выглядят разнородно. В кластере 0 — разные типы (седаны, внедорожники, минивэн, мини-автобус) и ракурсы. В кластере 2 — смесь седанов, внедорожников и хэтчбеков, белые и тёмные, в основном вид сверху. В кластере 4 много белых машин и вид сзади сверху, но типы кузова по-прежнему разные. Общего чёткого профиля по типу или ракурсу нет — группировка слабая.

**osnet:** В кластере 0 смешаны красные, серебристые, синие и жёлтые машины, седаны, внедорожники и автобус — единого признака не видно. В кластерах 2 и 4 группировка понятнее: кластер 2 — в основном белые и светлые машины, вид сзади сверху; кластер 4 — белые и серебристые, вид спереди сверху. То есть по ракурсу и светлому цвету osnet частично выделяет однородные группы.

**vdc_color:** Профиль по цвету читается чётко. Кластер 0 — тёмные машины (тёмно-синие, чёрные, серые седаны и внедорожники). Кластер 2 — тёплые тона (красные, жёлтые, оранжевые машины разных типов). Кластер 4 — светлые (серебристые, серые, белые). Дескриптор vdc_color действительно группирует по цвету.

**vdc_type:** В кластере 0 — смесь седанов, минивэна и внедорожника, без единого типа. В кластере 2 — только легковые седаны, вид сзади. В кластере 4 — лёгкие грузовики и коммерческий транспорт (пикапы, бортовые, фургоны с надписями FORLAND, FOTON, DFAC и т.п.). То есть vdc_type в кластерах 2 и 4 даёт понятный тип (седаны сзади и коммерческий транспорт).

Итог: Специализированные дескрипторы **vdc_color** и **vdc_type** дают интерпретируемые кластеры (цвет и тип/класс ТС). **osnet** — частично (ракурс и светлые машины в части кластеров). **efficientnet-b7** — группировка наименее понятная, однородность по типу или ракурсу слабая.


### 3.3. Поиск выбросов


С помощью известных вам методов поиска выбросов (например, DBSCAN) попытайтесь найти выбросы среди изображений, используя все варианты дескрипторов. Подберите параметры алгоритма.

Визуализируйте изображения, попавшие в раздел выбросов, и попробуйте проинтерпретировать полученные результаты. Подумайте, почему именно эти изображения попали в выбросы.

Сравните результаты для всех вариантов дескрипторов. Какой вариант дескрипторов даёт наилучшее представление о выбросах?


In [ ]:
# =============================================================================
# 3.3. Поиск выбросов (DBSCAN по подвыборке 25k — экономия памяти)
# Метка -1 = выброс. Визуализация и таблица — по этой подвыборке.
# =============================================================================

from sklearn.cluster import DBSCAN

np.random.seed(42)
OUTLIER_SAMPLE_SIZE = 25_000  # DBSCAN по подвыборке, чтобы не ронять ядро

BASE_DIR = Path('..') / 'data'
IMAGES_BASE = BASE_DIR / 'raw_data'
SCALING_USED = 'std'

MIN_SAMPLES = 5
EPS_BY_DESCRIPTOR = {'efficientnet-b7': 55.0, 'osnet': 15.0, 'vdc_color': 14.0, 'vdc_type': 12.0}

outlier_results = {}

print("Поиск выбросов DBSCAN (по подвыборке для экономии памяти)...")
for name in tqdm(preprocessed.keys(), desc="Дескрипторы"):
    X_full = preprocessed[name][SCALING_USED]['X']
    paths_full = preprocessed[name][SCALING_USED]['paths']
    n = len(paths_full)
    n_use = min(OUTLIER_SAMPLE_SIZE, n)
    idx = np.random.RandomState(42).choice(n, n_use, replace=False)
    X = X_full[idx]
    paths = [paths_full[i] for i in idx]
    full_paths = [str(IMAGES_BASE / str(p).replace('\\', '/')) for p in paths]

    eps = EPS_BY_DESCRIPTOR.get(name, 15.0)
    model = DBSCAN(eps=eps, min_samples=MIN_SAMPLES, metric='euclidean', algorithm='ball_tree', n_jobs=1)
    labels = model.fit_predict(X)
    n_outliers = (labels == -1).sum()

    outlier_results[name] = {
        'data': pd.DataFrame({'paths': full_paths, 'cluster': labels}),
        'labels': labels,
        'n_outliers': n_outliers,
        'n_total': n_use,
    }
    print(f"  {name}: выбросов {n_outliers} из {n_use} ({100 * n_outliers / n_use:.2f}%), eps={eps}")

print("Готово.") 


In [ ]:
# Визуализация по 9 случайных изображений-выбросов для каждого дескриптора
np.random.seed(42)
NROWS, NCOLS = 3, 3

for name in tqdm(outlier_results.keys(), desc="Визуализация выбросов"):
    data = outlier_results[name]['data']
    n_out = outlier_results[name]['n_outliers']
    if n_out == 0:
        print(f"{name}: выбросов нет, пропуск.")
        continue
    plot_samples_images(data, cluster_label=-1, nrows=NROWS, ncols=NCOLS, figsize=(12, 5))
    plt.gcf().suptitle(f"{name} — выбросы (всего {n_out})", fontsize=16)
    plt.show() 


In [ ]:
# Сводка: дескриптор, число выбросов, доля
rows = [
    {
        'Дескриптор': name,
        'Число выбросов': outlier_results[name]['n_outliers'],
        'Всего объектов': outlier_results[name]['n_total'],
        'Доля выбросов, %': round(100 * outlier_results[name]['n_outliers'] / outlier_results[name]['n_total'], 2),
    }
    for name in outlier_results
]
table_outliers = pd.DataFrame(rows)
display(table_outliers) 


**Вывод**

- Поиск выбросов выполнен алгоритмом DBSCAN (метка -1) по подвыборке из 25 000 объектов для каждого дескриптора из-за ограничений по памяти. Параметры eps заданы отдельно по дескриптору с учётом масштаба пространства после PCA.

- По доле выбросов: **osnet** даёт 23,78% (5946 из 25 000) — существенно больше остальных; **efficientnet-b7** — 0,70% (176), **vdc_type** — 1,29% (323), **vdc_color** — 0,03% (7). То есть osnet помечает как выбросы много объектов, efficientnet и vdc_type — умеренное число, vdc_color — почти ничего.

- Наиболее богатое множество выбросов для визуальной проверки даёт **osnet**. По снимкам, попавшим в выбросы, можно проверить, есть ли среди них кадры без машин, с сильными бликами или плохим качеством. У **efficientnet-b7** и **vdc_type** выбросов меньше, но их тоже имеет смысл просмотреть и описать. **vdc_color** при текущем eps почти не выделяет выбросов — пространство признаков цвета, видимо, очень плотное.

- Итог: для автоматизированного поиска выбросов с последующей ручной проверкой удобнее всего использовать **osnet** (много кандидатов) или комбинацию **osnet** и **vdc_type**. В отчёте и при сохранении результатов стоит явно указать, что разметка выбросов получена по подвыборке 25k и перенесена на полные данные только при необходимости повторным запуском с той же логикой.


## 4. Выводы и экспорт результатов


In [ ]:
from pathlib import Path

# =============================================================================
# 4. Сводная таблица кластеризации и сохранение лучшего результата в CSV
# Лучший вариант: vdc_type + AgglomerativeClustering.
# CSV сохраняем в папку data.
# =============================================================================

BEST_DESCRIPTOR = 'vdc_type'
BEST_ALGORITHM = 'AgglomerativeClustering'

# Путь к папке data 
BASE_DIR = Path('..') / 'data'
OUTPUT_CSV = BASE_DIR / 'clustering_results_best.csv'

# Таблица сравнения кластеризации (из раздела 3.1)
display(table_clustering)

# Сохранение лучшей разметки: path, cluster
paths_best = preprocessed[BEST_DESCRIPTOR]['std']['paths']
labels_best = clustering_results[BEST_DESCRIPTOR][BEST_ALGORITHM]['labels']
df_export = pd.DataFrame({'path': paths_best, 'cluster': labels_best})
df_export['path'] = df_export['path'].astype(str).str.replace('\\', '/', regex=False)
df_export.to_csv(OUTPUT_CSV, index=False)
print(f"Сохранено: {OUTPUT_CSV}, строк: {len(df_export)}") 


**Итоговые выводы по проекту**

- По внутренним метрикам (Калински — Харабаса и Дэвиса — Болдина) и визуализации t-SNE наилучшее качество кластеризации дают дескрипторы **vdc_type** и **vdc_color**: их кластеры наиболее хорошо разделены в пространстве признаков и наиболее легко интерпретируются визуально. Для **osnet** кластеры разделены средне, для **efficientnet-b7** — хуже всего, кластеры часто смешивают разные типы и ракурсы.

- Анализ примеров изображений по кластерам показал, что для **vdc_type** удаётся разделить автомобили по типу и назначению: отдельные кластеры соответствуют легковым седанам, кроссоверам/минивэнам, пассажирским минивэнам/микроавтобусам и лёгким грузовикам/коммерческому транспорту. Для **vdc_color** кластеры хорошо группируются по цвету (тёмные, светлые, красные/жёлтые и т.п.). Для **osnet** часть кластеров отражает комбинации ракурса и цвета (например, белые машины с видом сзади/спереди), но общая структура менее однозначна. Для **efficientnet-b7** кластеры визуально наиболее неоднородны: в одном кластере часто оказываются разные типы кузова, ракурсы и цвета, что подтверждает более слабое качество признакового пространства этого дескриптора для задач кластеризации.

- Поиск выбросов методом **DBSCAN** (на подвыборке по 25 000 объектов для каждого дескриптора) показал, что доля выбросов сильно зависит от дескриптора: для vdc_color и vdc_type выбросов очень мало (0.03% и 1.29% соответственно), для efficientnet-b7 — <1%, тогда как для osnet около 24%. Визуальный анализ примеров выбросов показывает, что они часто соответствуют редким конфигурациям: необычным типам ТС (автобусы, специальные грузовики), нетипичным цветам или ракурсам, а также кадрам с частичными перекрытиями или сильным отличием фона. Это подчёркивает, что «выброс» в терминах DBSCAN — не обязательно «плохое изображение», а объект, который слабо похож на плотные группы в признаковом пространстве.

- В качестве **основной конфигурации для задач разметки** можно рекомендовать дескриптор **vdc_type** в сочетании с алгоритмом **AgglomerativeClustering** (k=5) и предобработкой StandardScaler + PCA (90% дисперсии). Такая связка даёт хорошо разделимые и интерпретируемые кластеры по типу транспортного средства и была использована для формирования итогового CSV с полями `path` и `cluster`. Для анализа по цвету полезно дополнять её дескриптором **vdc_color**, а для поиска кандидатов на выбросы — использовать дескриптор **osnet** с DBSCAN и последующей визуальной проверкой.


**Расшифровка кластеров (vdc_type + AgglomerativeClustering, k=5)**

- **Кластер 0:** в основном кроссоверы и минивэны, встречаются также пикапы и седаны; ракурсы смешанные (вид спереди и сзади сверху).

- **Кластер 1:** преимущественно минивэны и микробусы (много белых/серебристых машин), местами небольшие фургоны; ракурс в основном сзади/сбоку сверху.

- **Кластер 2:** почти все объекты — легковые седаны разных цветов, в основном вид сзади сверху.

- **Кластер 3:** смесь седанов и кроссоверов/паркетников без грузового транспорта; ракурсы преимущественно спереди/сбоку сверху.

- **Кластер 4:** лёгкие грузовики и коммерческий транспорт (пикапы, бортовые, небольшие фургоны); ракурсы преимущественно сзади/сбоку сверху.

Итоговая разметка сохранена в файле `clustering_results_best.csv` (столбцы `path`, `cluster`).
